# RPC demo: reflection-position constrained result

Noiseless demo using `seis.mat`, `ref.mat`, and the provided noiseless position-constrained prediction `alm_predict_3d.mat`. The notebook plots the true reflection positions, predicted/true position overlay, and RPC wiggle result.

In [ ]:
EPOCHS = 10
PROFILE_TRACE = 100
TRACE_START = 0
TRACE_COUNT = 30
TIME_START = 200
TIME_STOP = 800
DT = 0.001
T0 = TIME_START * DT


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from demo_utils import (
    DATA_DIR,
    FIGURE_DIR,
    RESULT_DIR,
    extract_positions,
    load_mat_array,
    pick_interval_section,
    plot_position_points,
    save_mat,
    sparse_spike_deconvolution,
    to_interval_trace_time,
    train_position_demo,
    train_reflectivity_demo,
    wigb,
)

plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 14,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 0.9,
    "savefig.dpi": 300,
})
FIGURE_DIR.mkdir(exist_ok=True)
RESULT_DIR.mkdir(exist_ok=True)


## Load noiseless data

In [ ]:
ref = to_interval_trace_time(load_mat_array(DATA_DIR / "ref.mat", "ref"))
seis = to_interval_trace_time(load_mat_array(DATA_DIR / "seis.mat", "seis"))
rpc = load_mat_array(DATA_DIR / "alm_predict_3d.mat", "pre_alm")

ref_sec = pick_interval_section(ref, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
seis_sec = pick_interval_section(seis, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
rpc_sec = pick_interval_section(rpc, PROFILE_TRACE, TRACE_START, TRACE_COUNT, TIME_START, TIME_STOP)
print("ref", ref.shape, "seis", seis.shape, "rpc", rpc.shape)

## Paper-style RPC figure

In [ ]:
true_pos = extract_positions(ref_sec, threshold=1e-20)
rpc_threshold = max(1e-8, 0.08 * np.nanmax(np.abs(rpc_sec)))
pred_pos = extract_positions(rpc_sec, threshold=rpc_threshold)

fig, axes = plt.subplots(2, 2, figsize=(10, 9), constrained_layout=True)
plot_position_points(true_pos, ax=axes[0, 0], color="black", size=8, panel_label="(a)", dt=DT, t0=T0)

time_true, trace_true = np.nonzero(true_pos)
time_pre, trace_pre = np.nonzero(pred_pos)
axes[0, 1].scatter(trace_true + 1, (T0 + time_true * DT) * 1000, c="red", s=10, marker=".", linewidths=0)
axes[0, 1].scatter(trace_pre + 1, (T0 + time_pre * DT) * 1000, c="blue", s=10, marker=".", linewidths=0)
axes[0, 1].set_xlim(0.5, TRACE_COUNT + 0.5)
axes[0, 1].set_ylim(TIME_STOP, TIME_START)
axes[0, 1].set_xlabel("Trace no.")
axes[0, 1].set_ylabel("Time (ms)")
axes[0, 1].tick_params(direction="in", top=True, right=True)
axes[0, 1].text(-0.12, 1.02, "(b)", transform=axes[0, 1].transAxes, fontsize=16)

wigb(seis_sec, dt=DT, t0=T0, scale=0.75, ax=axes[1, 0], linewidth=0.6, panel_label="(c)")
wigb(rpc_sec, dt=DT, t0=T0, scale=0.75, ax=axes[1, 1], linewidth=0.6, panel_label="(d)")
selected_trace = 13
axes[1, 1].axvline(selected_trace, color="red", linewidth=1.4)
axes[1, 1].text(selected_trace, TIME_STOP + 18, str(selected_trace), color="red", ha="center", va="top", fontsize=16)
fig.savefig(FIGURE_DIR / "rpc_demo.png", bbox_inches="tight")
plt.show()